# F20: Noise Talk — Information-Theoretic Metrics

Purpose-built analyses for "The Statistical Unconscious" (Noisy Systems, Rome, 4 June 2026).

1. **Character-level redundancy** — Shannon's redundancy from BLT bits/char
2. **KL divergence** — alignment-as-noise in bits per token, from cached logits
3. **Summary figure** — Shannon channel model with actual numbers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150, 'figure.figsize': (10, 6)})

FIG_DIR = '../figures'
NOISE_FIG_DIR = '/Users/rj416/Dropbox/Prof/Confs/Noise2026/figures'

blt = pd.read_csv('../data/blt_combined.csv')
shannon = pd.read_csv('../data/shannon_entropy.csv')
self_surp = pd.read_csv('../data/self_surprisal.csv')
logit_metrics = pd.read_csv('../data/logit_metrics.csv')

print(f"BLT: {len(blt):,} rows, sources: {sorted(blt.source.unique())}")
print(f"Shannon entropy: {len(shannon)} rows")
print(f"Self-surprisal: {len(self_surp)} rows")
print(f"Logit metrics: {len(logit_metrics)} rows")

## 1. Character-level redundancy (Shannon 1951)

Shannon measured English at ~1.0 bits/char, against a maximum of log2(27) ≈ 4.75 bits/char (26 letters + space). Redundancy = 1 - H/H_max ≈ 79%.

We compute the same from BLT bits/char for base models, aligned models, and human genres.

In [ ]:
H_MAX_CHAR = np.log2(27)  # Shannon's alphabet: 26 letters + space

AI_FAMILIES = ['olmo', 'olmo-tiny', 'llama', 'qwen', 'qwen-tiny', 'amber', 
               'tulu', 'zephyr', 'pythia', 'smol']
HUMAN_SOURCES = ['dreams', 'waking', 'c20_fiction', 'abstracts']

# Classify each row
blt['is_ai'] = blt.source.isin(AI_FAMILIES)
blt['is_human'] = blt.source.isin(HUMAN_SOURCES)

# Filter to prose only for AI (BOS generations), all for human
ai_bos = blt[(blt.is_ai) & (blt.prompt_type == 'bos')].copy()
human = blt[blt.is_human].copy()

# Classify layer as base or aligned
def classify_layer(row):
    if row['layer'] == 'base':
        return 'Base model'
    elif row['layer'] in ('sft', 'dpo', 'instruct', 'rlvr'):
        return 'Aligned model'
    return None

ai_bos['group'] = ai_bos.apply(classify_layer, axis=1)
ai_bos = ai_bos.dropna(subset=['group'])
human['group'] = 'Human text'

# Compute medians
groups = {}
for name, grp in ai_bos.groupby('group'):
    groups[name] = grp.bits_per_char.median()
for name, grp in human.groupby('source'):
    groups[name] = grp.bits_per_char.median()

# Redundancy
print(f"Shannon's H_max (27-char alphabet): {H_MAX_CHAR:.2f} bits/char")
print(f"Shannon's measured H (1951):         1.00 bits/char")
print(f"Shannon's redundancy:                {1 - 1.0/H_MAX_CHAR:.1%}")
print()
print(f"{'Group':<20} {'Median bits/char':>16} {'Redundancy':>12}")
print("-" * 52)
for name in ['Base model', 'Aligned model', 'dreams', 'waking', 'c20_fiction', 'abstracts']:
    if name in groups:
        h = groups[name]
        r = 1 - h / H_MAX_CHAR
        print(f"{name:<20} {h:>16.3f} {r:>12.1%}")

## 2. KL divergence: alignment-as-noise in bits per token

For each prompt × family, compute KL(base || aligned) from the cached full-vocabulary logit distributions. This literally measures how many bits of information alignment destroys per token — the "noise" of the alignment channel in Shannon's framework.

Also compute JS divergence in bits and entropy reduction in bits for completeness.

In [ ]:
import torch
from malign_logits.cache import CacheManager
from malign_logits import MODEL_FAMILIES
from malign_logits.experiments import DEFAULT_PROMPTS

cm = CacheManager()
LN2 = np.log(2)
EPS = 1e-10

def safe_probs(logits, n):
    """Softmax with epsilon floor to prevent log(0)."""
    p = torch.softmax(torch.tensor(logits[:n], dtype=torch.float64), dim=-1)
    p = p.clamp(min=EPS)
    p = p / p.sum()
    return p

def kl_div_bits(logits_p, logits_q):
    n = min(len(logits_p), len(logits_q))
    p = safe_probs(logits_p, n)
    q = safe_probs(logits_q, n)
    return (p * (p.log() - q.log())).sum().item() / LN2

def js_div_bits(logits_p, logits_q):
    n = min(len(logits_p), len(logits_q))
    p = safe_probs(logits_p, n)
    q = safe_probs(logits_q, n)
    m = 0.5 * (p + q)
    return (0.5 * ((p * (p.log() - m.log())).sum() + (q * (q.log() - m.log())).sum())).item() / LN2

rows = []
for fam_key, fam in MODEL_FAMILIES.items():
    base_id = fam.base
    aligned_id = fam.superego
    if aligned_id is None:
        continue
    
    for label, prompt in DEFAULT_PROMPTS.items():
        category = label.rsplit('_', 1)[0]
        
        if not cm.has_logits(base_id, prompt) or not cm.has_logits(aligned_id, prompt):
            continue
        
        base_logits = cm.get_logits(base_id, prompt)
        aligned_logits = cm.get_logits(aligned_id, prompt)
        
        rows.append({
            'family': fam_key, 'prompt': prompt, 'label': label, 'category': category,
            'kl_base_aligned_bits': kl_div_bits(base_logits, aligned_logits),
            'kl_aligned_base_bits': kl_div_bits(aligned_logits, base_logits),
            'js_bits': js_div_bits(base_logits, aligned_logits),
        })

kl_df = pd.DataFrame(rows)
print(f"Computed KL for {len(kl_df)} prompt×family pairs")
print(f"Families: {sorted(kl_df.family.unique())}")
print(f"Any inf? {np.isinf(kl_df.kl_base_aligned_bits).sum()}")
kl_df.to_csv('../data/kl_divergence.csv', index=False)
print("Saved data/kl_divergence.csv")

In [ ]:
# Summarize KL results
print("=== Alignment noise by family (KL divergence, bits per token) ===")
fam_kl = kl_df.groupby('family').kl_base_aligned_bits.agg(['mean', 'median', 'std']).round(3)
fam_kl = fam_kl.sort_values('mean', ascending=False)
print(fam_kl.to_string())

print("\n=== Alignment noise by content category (KL, bits, all families) ===")
cat_kl = kl_df.groupby('category').kl_base_aligned_bits.agg(['mean', 'median', 'count']).round(3)
cat_kl = cat_kl.sort_values('mean', ascending=False)
print(cat_kl.to_string())

print(f"\nOverall: alignment destroys {kl_df.kl_base_aligned_bits.mean():.2f} bits/token (mean)")
print(f"         {kl_df.kl_base_aligned_bits.median():.2f} bits/token (median)")

### 2b. Per-family breakdown: KL divergence and BLT bits/char

In [ ]:
# --- Per-family BLT bits/char (base vs aligned, BOS prose) ---
blt_ai = blt[(blt.source.isin(AI_FAMILIES)) & (blt.prompt_type == 'bos')].copy()
blt_ai['group'] = blt_ai.layer.apply(lambda x: 'Base' if x == 'base' else 'Aligned')

blt_fam = blt_ai.groupby(['source', 'group']).bits_per_char.median().unstack(fill_value=np.nan)
blt_fam['delta'] = blt_fam.get('Base', 0) - blt_fam.get('Aligned', 0)
blt_fam = blt_fam.sort_values('delta', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: base vs aligned by family
ax = axes[0]
fams = blt_fam.index.tolist()
y = np.arange(len(fams))
if 'Base' in blt_fam.columns and 'Aligned' in blt_fam.columns:
    ax.barh(y - 0.15, blt_fam['Base'], height=0.3, color='#27ae60', label='Base', alpha=0.8)
    ax.barh(y + 0.15, blt_fam['Aligned'], height=0.3, color='#e74c3c', label='Aligned', alpha=0.8)
ax.axvline(1.0, color='#2c3e50', ls='--', alpha=0.4, label='Shannon 1.0')
ax.set_yticks(y)
ax.set_yticklabels(fams)
ax.set_xlabel('BLT bits/char')
ax.set_title('Information density by family')
ax.legend(fontsize=9)
ax.invert_yaxis()

# Right: KL divergence by family
ax = axes[1]
fam_order = kl_df.groupby('family').kl_base_aligned_bits.median().sort_values(ascending=False)
fams_kl = fam_order.index.tolist()
y = np.arange(len(fams_kl))
vals = [fam_order[f] for f in fams_kl]
ax.barh(y, vals, color='#8e44ad', alpha=0.8)
for i, v in enumerate(vals):
    ax.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=10)
ax.set_yticks(y)
ax.set_yticklabels(fams_kl)
ax.set_xlabel('KL(base || aligned) bits/token')
ax.set_title('Alignment noise by family (median)')
ax.invert_yaxis()

plt.tight_layout()
for path in [f'{FIG_DIR}/F20_per_family_noise.png', f'{NOISE_FIG_DIR}/F20_per_family_noise.png']:
    fig.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved {path}')
plt.show()

In [ ]:
# --- KL divergence heatmap: family × category ---
pivot = kl_df.pivot_table(
    index='family', columns='category', values='kl_base_aligned_bits', aggfunc='median'
)
# Sort families by overall median KL
fam_medians = kl_df.groupby('family').kl_base_aligned_bits.median().sort_values(ascending=False)
pivot = pivot.loc[fam_medians.index]

# Sort categories
cat_order = ['sexual_explicit', 'sexual_liminal', 'violence_explicit', 'violence_liminal',
             'death', 'power', 'profanity', 'substance', 'neutral']
pivot = pivot[[c for c in cat_order if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([c.replace('_', '\n') for c in pivot.columns], fontsize=10, rotation=0)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=11)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            color = 'white' if v > pivot.values[~np.isnan(pivot.values)].mean() else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9, color=color)

ax.set_title('Alignment noise: KL(base || aligned) bits/token by family × category', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='KL divergence (bits)', shrink=0.8)
plt.tight_layout()
for path in [f'{FIG_DIR}/F20_kl_heatmap.png', f'{NOISE_FIG_DIR}/F20_kl_heatmap.png']:
    fig.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved {path}')
plt.show()

## 3. Summary figure: Shannon's channel model, quantified

A single figure mapping alignment onto Shannon's model with actual numbers. Shows:
- Source entropy (base model bits/char via BLT)
- Channel noise (alignment delta)  
- Output entropy (aligned model bits/char)
- Human genres for reference
- Shannon's 1951 measurement as benchmark

In [ ]:
# Compute median BLT bits/char for each group
ai_families = ['olmo', 'olmo-tiny', 'llama', 'qwen', 'qwen-tiny', 'amber', 
               'tulu', 'zephyr', 'pythia', 'smol']

blt_bos = blt[(blt.source.isin(ai_families)) & (blt.prompt_type == 'bos')].copy()
blt_human = blt[blt.source.isin(['dreams', 'waking', 'c20_fiction', 'abstracts'])].copy()

base_bpc = blt_bos[blt_bos.layer == 'base'].bits_per_char
aligned_bpc = blt_bos[blt_bos.layer != 'base'].bits_per_char

data = {
    'Aligned models': aligned_bpc.median(),
    'Shannon English (1951)': 1.0,
    'Base models': base_bpc.median(),
    'Waking journals': blt_human[blt_human.source == 'waking'].bits_per_char.median(),
    'Abstracts': blt_human[blt_human.source == 'abstracts'].bits_per_char.median(),
    'Dreams': blt_human[blt_human.source == 'dreams'].bits_per_char.median(),
    'Fiction': blt_human[blt_human.source == 'c20_fiction'].bits_per_char.median(),
}

fig, ax = plt.subplots(figsize=(10, 5))

colors = {
    'Aligned models': '#e74c3c',
    'Base models': '#27ae60',
    'Shannon English (1951)': '#2c3e50',
    'Waking journals': '#3498db',
    'Abstracts': '#3498db',
    'Dreams': '#3498db',
    'Fiction': '#3498db',
}

y_pos = list(range(len(data)))
labels = list(data.keys())
values = list(data.values())

bars = ax.barh(y_pos, values, color=[colors[l] for l in labels], height=0.6, edgecolor='white', linewidth=0.5)

for i, (label, val) in enumerate(zip(labels, values)):
    ax.text(val + 0.03, i, f'{val:.2f}', va='center', fontsize=11, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel('BLT bits/char (byte-level cross-entropy)', fontsize=12)
ax.set_title('Information density: alignment compresses below Shannon\'s English rate', fontsize=13, fontweight='bold')
ax.axvline(x=1.0, color='#2c3e50', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlim(0, max(values) * 1.25)
ax.invert_yaxis()

# Add annotation
delta = data['Base models'] - data['Aligned models']
ax.annotate(f'Alignment removes\n{delta:.2f} bits/char',
            xy=(data['Aligned models'], 0), xytext=(data['Base models'] + 0.15, 1.5),
            fontsize=10, ha='left',
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
for path in [f'{FIG_DIR}/F20_shannon_channel.png', f'{NOISE_FIG_DIR}/F20_shannon_channel.png']:
    fig.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved {path}')
plt.show()

## 4. Export summary for talk

In [ ]:
# Key numbers for the talk
H_MAX = np.log2(27)
print("=" * 60)
print("NUMBERS FOR THE TALK")
print("=" * 60)

print(f"\n--- Shannon reference ---")
print(f"  H_max (27-char alphabet):  {H_MAX:.2f} bits/char")
print(f"  Shannon's English (1951):  1.00 bits/char")
print(f"  Shannon's redundancy:      {1 - 1.0/H_MAX:.0%}")

print(f"\n--- BLT bits/char (median, BOS prose) ---")
for name, val in sorted(data.items(), key=lambda x: x[1]):
    r = 1 - val / H_MAX
    print(f"  {name:<25s} {val:.2f} bits/char  (redundancy {r:.0%})")

print(f"\n--- KL divergence: alignment noise (bits/token, mean per family) ---")
for _, row in fam_kl.iterrows():
    print(f"  {row.name:<12s} {row['mean']:.2f} bits/token")

print(f"\n--- KL by category (bits/token, mean across families) ---")
for _, row in cat_kl.iterrows():
    print(f"  {row.name:<25s} {row['mean']:.2f} bits")

print(f"\n--- Uniform compression (Kruskal-Wallis on entropy reduction) ---")
from scipy.stats import kruskal
lm_ent = logit_metrics.copy()
lm_ent['ent_delta'] = lm_ent['entropy_base'] - lm_ent['entropy_superego'].fillna(lm_ent['entropy_instruct'])
cats = lm_ent.dropna(subset=['ent_delta']).groupby('category').ent_delta.apply(list).to_dict()
if len(cats) > 2:
    stat, p = kruskal(*cats.values())
    print(f"  H={stat:.2f}, p={p:.4f} across {len(cats)} categories")
    print(f"  → {'Content category has NO effect on compression' if p > 0.05 else 'Content category matters'}")

## 5. Self-surprisal boxplots: three measures, one pattern (v4, matplotlib)

Three-panel figure: self-surprisal | Pythia 1B reference | BLT 1B reference.
Each AI family gets base/aligned boxplots; each human genre gets one boxplot.
BOS prose only — unconditional generation filtered to prose + English.

In [ ]:
from matplotlib.patches import Patch

jak = pd.read_parquet('../data/jakobson.parquet')

# AI: BOS + prose + English only
ai_jak = jak[(jak['corpus_type'] == 'ai') & (jak['prompt_type'] == 'bos') & 
             (jak['genre'] == 'prose') & (jak['language'] == 'en')]
base_jak = ai_jak[ai_jak['layer'] == 'base']
aligned_jak = ai_jak[ai_jak['layer'] == 'superego']
human_jak = jak[jak['corpus_type'] == 'human']

# Family order by base self-surprisal median
common_fams = set(base_jak['family'].unique()) & set(aligned_jak['family'].unique())
fam_order = (base_jak[base_jak['family'].isin(common_fams)]
             .groupby('family')['self_bits_per_char'].median()
             .sort_values(ascending=True).index.tolist())
human_order = human_jak.groupby('family')['blt_bits_per_char'].median().sort_values(ascending=True).index.tolist()
human_labels = {'abstracts': 'Abstracts', 'waking': 'Journals', 'fiction': 'Fiction', 'dreams': 'Dreams'}

panels = [
    ('Self-surprisal\n(model scores own output)', 'self_bits_per_char', True),
    ('Pythia 1B reference', 'ref_bits_per_char', False),
    ('BLT 1B reference', 'blt_bits_per_char', False),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 10))
spacing = 2.5
bar_offset = 0.4

# Build shared y-positions
yticks_all, ylabels_all = [], []
y = 0
for genre in reversed(human_order):
    yticks_all.append(y); ylabels_all.append(human_labels.get(genre, genre)); y += spacing
sep_y_pos = y - spacing/2 + 0.4
y += 0.6
for fam in reversed(fam_order):
    yticks_all.append(y); ylabels_all.append(fam); y += spacing
ymax = y

for ax_idx, (title, col, is_self) in enumerate(panels):
    ax = axes[ax_idx]
    y = 0
    for genre in reversed(human_order):
        if is_self:
            ax.text(1.0, y, 'n/a', fontsize=9, color='#aaaaaa', ha='center', va='center')
        else:
            d = human_jak[human_jak['family'] == genre][col].dropna().values
            if len(d) > 0:
                ax.boxplot([d], positions=[y], vert=False, widths=0.6, patch_artist=True, showfliers=False,
                           boxprops=dict(facecolor='#7eb3e8', alpha=0.7, edgecolor='#2a5f8f'),
                           medianprops=dict(color='#1a3d5c', linewidth=2),
                           whiskerprops=dict(color='#2a5f8f'), capprops=dict(color='#2a5f8f'))
        y += spacing
    ax.axhline(y=sep_y_pos, color='#999999', linestyle='-', linewidth=0.8, alpha=0.5)
    y += 0.6
    for fam in reversed(fam_order):
        bd = base_jak[base_jak['family'] == fam][col].dropna().values
        ad = aligned_jak[aligned_jak['family'] == fam][col].dropna().values
        if len(bd) > 0:
            ax.boxplot([bd], positions=[y + bar_offset], vert=False, widths=0.55, patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor='#8ecf8e', alpha=0.7, edgecolor='#3a7d3a'),
                       medianprops=dict(color='#2d5a2d', linewidth=2),
                       whiskerprops=dict(color='#3a7d3a'), capprops=dict(color='#3a7d3a'))
        if len(ad) > 0:
            ax.boxplot([ad], positions=[y - bar_offset], vert=False, widths=0.55, patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor='#e88e8e', alpha=0.7, edgecolor='#a03a3a'),
                       medianprops=dict(color='#6d1a1a', linewidth=2),
                       whiskerprops=dict(color='#a03a3a'), capprops=dict(color='#a03a3a'))
        y += spacing
    ax.axvline(x=1.0, color='#333333', linestyle='--', linewidth=1.5, zorder=0, alpha=0.6)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Bits per character', fontsize=11)
    ax.set_xlim(0, 2.5); ax.set_ylim(-1.5, ymax + 0.5)
    ax.set_yticks(yticks_all)
    ax.set_yticklabels(ylabels_all if ax_idx == 0 else [], fontsize=11)

axes[1].legend(
    [Patch(facecolor='#8ecf8e', alpha=0.7, edgecolor='#3a7d3a'),
     Patch(facecolor='#e88e8e', alpha=0.7, edgecolor='#a03a3a'),
     Patch(facecolor='#7eb3e8', alpha=0.7, edgecolor='#2a5f8f')],
    ['Base model', 'Aligned model', 'Human text'], fontsize=10, loc='upper right')
axes[0].text(1.03, ymax - 0.5, "Shannon\n≈ 1.0", fontsize=9, color='#333333', va='top')

fig.suptitle('Information density (BOS prose): three measures, one pattern', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
for path in [f'{FIG_DIR}/F18_self_surprisal_boxplot_v4.png', f'{NOISE_FIG_DIR}/F18_self_surprisal_boxplot_v4.png']:
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved {path}')
plt.show()

## 6. Self-surprisal boxplots v5 (plotnine, dpi=300)

Same three-panel layout as v4, rendered with plotnine/ggplot for cleaner typography and publication-quality output.

In [ ]:
from plotnine import *
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

jak = pd.read_parquet('../data/jakobson.parquet')

# AI: BOS + prose + English
ai_p = jak[(jak['corpus_type'] == 'ai') & (jak['prompt_type'] == 'bos') & 
           (jak['genre'] == 'prose') & (jak['language'] == 'en')].copy()
base_p = ai_p[ai_p['layer'] == 'base'].copy()
aligned_p = ai_p[ai_p['layer'] == 'superego'].copy()
human_p = jak[jak['corpus_type'] == 'human'].copy()

common = set(base_p['family'].unique()) & set(aligned_p['family'].unique())
fam_ord = (base_p[base_p['family'].isin(common)]
           .groupby('family')['self_bits_per_char'].median()
           .sort_values().index.tolist())
human_labels = {'abstracts': 'Abstracts', 'waking': 'Journals', 'fiction': 'Fiction', 'dreams': 'Dreams'}
human_ord = (human_p.groupby('family')['blt_bits_per_char'].median()
             .sort_values().index.tolist())

# Y-order: AI families (high surprisal top) then human genres (low BLT top)
y_order = list(reversed(fam_ord)) + [human_labels[g] for g in reversed(human_ord)]

metrics = {
    'Self-surprisal': 'self_bits_per_char',
    'Pythia 1B': 'ref_bits_per_char',
    'BLT 1B': 'blt_bits_per_char',
}

rows = []
for metric_name, col in metrics.items():
    for fam in fam_ord:
        for layer_df, group in [(base_p, 'Base'), (aligned_p, 'Aligned')]:
            vals = layer_df[layer_df['family'] == fam][col].dropna().values
            for val in vals:
                rows.append({'metric': metric_name, 'y_label': fam, 'group': group, 'value': val})
    if metric_name != 'Self-surprisal':
        for genre in human_ord:
            vals = human_p[human_p['family'] == genre][col].dropna().values
            for val in vals:
                rows.append({'metric': metric_name, 'y_label': human_labels[genre], 
                            'group': 'Human text', 'value': val})

plot_df = pd.DataFrame(rows)
plot_df['y_label'] = pd.Categorical(plot_df['y_label'], categories=y_order, ordered=True)
plot_df['group'] = pd.Categorical(plot_df['group'], categories=['Base', 'Aligned', 'Human text'], ordered=True)
plot_df['metric'] = pd.Categorical(plot_df['metric'], 
                                    categories=['Self-surprisal', 'Pythia 1B', 'BLT 1B'], ordered=True)

fill_map = {'Base': '#8ecf8e', 'Aligned': '#e88e8e', 'Human text': '#7eb3e8'}
edge_map = {'Base': '#3a7d3a', 'Aligned': '#a03a3a', 'Human text': '#2a5f8f'}

p = (ggplot(plot_df, aes(x='value', y='y_label', fill='group'))
     + geom_boxplot(aes(color='group'), outlier_shape='',
                    position=position_dodge(width=0.9, preserve='single'),
                    width=0.85, alpha=0.7, size=0.6, fatten=2)
     + geom_vline(xintercept=1.0, linetype='dashed', color='#444444', alpha=0.5, size=0.6)
     + facet_wrap('~metric', nrow=1, scales='free_x')
     + scale_fill_manual(values=fill_map, name='')
     + scale_color_manual(values=edge_map, guide=None)
     + coord_cartesian(xlim=(0, 2.5))
     + scale_y_discrete(name='')
     + labs(title='Information density (BOS prose): three measures, one pattern',
            x='Bits per character')
     + theme_bw()
     + theme(
         figure_size=(16, 9),
         plot_title=element_text(size=14, weight='bold'),
         strip_text=element_text(size=12, weight='bold'),
         axis_text_y=element_text(size=11),
         axis_text_x=element_text(size=10),
         legend_position='top',
         legend_text=element_text(size=11),
         panel_spacing_x=0.25,
         dpi=300,
     )
)

for path in [f'{FIG_DIR}/F18_self_surprisal_boxplot_v5.png', f'{NOISE_FIG_DIR}/F18_self_surprisal_boxplot_v5.png']:
    p.save(path, dpi=300, verbose=False)
    print(f'Saved {path}')
p

## 7. All families + frontier: BLT bits/char (battery, prose only)

Same boxplot format as above but using **battery prompts** (47 prompted completions × 100 gens) instead of BOS. This includes frontier API models (DeepSeek, Haiku, Sonnet, GPT-4o-mini) which don't have BOS generations. Excludes raw (no-system-prompt) condition. Prose genre only.

In [ ]:
from plotnine import *
import warnings
warnings.filterwarnings('ignore')

jak = pd.read_parquet('../data/jakobson.parquet')

EXCLUDE_FAMILIES = {'olmo-tiny', 'qwen-tiny', 'smol', 'deepseek-7b', 'gemini-flash'}
RAW_LAYERS = {'raw'}
FRONTIER_FAMILIES = {'claude-haiku', 'claude-sonnet', 'deepseek', 'gpt-4o-mini'}

ai = jak[(jak['corpus_type'] == 'ai') & 
         (jak['prompt_type'] == 'battery') & 
         (jak['genre'] == 'prose') &
         (~jak['family'].isin(EXCLUDE_FAMILIES)) &
         (~jak['layer'].isin(RAW_LAYERS))].copy()
human = jak[jak['corpus_type'] == 'human'].copy()

combined = pd.concat([ai, human], ignore_index=True)
combined = combined.dropna(subset=['blt_bits_per_char'])

def classify(row):
    if row['corpus_type'] == 'human':
        return 'Human text'
    if row['family'] in FRONTIER_FAMILIES:
        return 'Frontier'
    if row['layer'] == 'base':
        return 'Base'
    return 'Aligned'

combined['group'] = combined.apply(classify, axis=1)

def make_label(row):
    if row['corpus_type'] == 'human':
        if row['family'] == 'hemingway':
            return 'Hemingway'
        if row['family'] == 'joyce':
            return 'Joyce'
        if row['family'] in ('anderson', 'mansfield'):
            if row['layer'] == 'basic':
                return 'Basic English (stories)'
            else:
                return 'Original English (stories)'
        return {
            'abstracts': 'Abstracts', 'waking': 'Waking journals', 
            'fiction': 'C20 fiction', 'dreams': 'Dream reports',
        }.get(row['family'], row['family'])
    if row['family'] in FRONTIER_FAMILIES:
        return row['family']
    if row['layer'] == 'base':
        return f"{row['family']} (base)"
    return f"{row['family']} (aligned)"

combined['y_label'] = combined.apply(make_label, axis=1)

label_medians = combined.groupby('y_label')['blt_bits_per_char'].median().sort_values(ascending=False)
combined['y_label'] = pd.Categorical(combined['y_label'], 
                                      categories=label_medians.index.tolist(), 
                                      ordered=True)
combined['group'] = pd.Categorical(combined['group'], 
                                    categories=['Base', 'Aligned', 'Frontier', 'Human text'], 
                                    ordered=True)

fill_map = {'Base': '#8ecf8e', 'Aligned': '#e88e8e', 'Frontier': '#d4a0e8', 'Human text': '#7eb3e8'}
edge_map = {'Base': '#3a7d3a', 'Aligned': '#a03a3a', 'Frontier': '#7b2d9e', 'Human text': '#2a5f8f'}

p = (ggplot(combined, aes(y='blt_bits_per_char', x='y_label', fill='group'))
     + geom_boxplot(aes(color='group'), outlier_alpha=0,
                    width=0.7, alpha=0.7, size=0.4, fatten=2)
     + geom_hline(yintercept=1.0, linetype='dashed', color='#444444', alpha=0.5, size=0.6)
     + scale_fill_manual(values=fill_map, name='')
     + scale_color_manual(values=edge_map, guide=None)
     + coord_flip(ylim=(0.5, 2.5))
     + labs(title='Information density: all families + frontier (battery prompts, prose)',
            subtitle='BLT 1B bits/char. Dashed line = Shannon\'s English rate (~1.0). Aligned = all post-training layers pooled.',
            y='BLT bits/char', x='')
     + theme_bw()
     + theme(
         figure_size=(12, 10),
         plot_title=element_text(size=13, weight='bold'),
         plot_subtitle=element_text(size=9),
         axis_text_y=element_text(size=9),
         axis_text_x=element_text(size=10),
         legend_position='top',
         legend_text=element_text(size=10),
         dpi=300,
     )
)

for path in [f'{FIG_DIR}/F20_blt_all_families_battery.png', f'{NOISE_FIG_DIR}/F20_blt_all_families_battery.png']:
    try:
        p.save(path, dpi=300, verbose=False)
        print(f'Saved {path}')
    except Exception as e:
        print(f'Skip {path}: {e}')
p